# 7-day propensity model (clean notebook)

Predict **`y_*_7d`** take-up for one green product. Uses the dataset's **train / val / test** column and **eligible customers only** (`elig_* = 1`).

- **Validation deciles:** equal **customer count** (~10% each), decile 1 = highest score  
- **Test deciles:** same score cutoffs as validation (counts may differ)  
- Install: `%pip install -q pandas numpy scikit-learn xgboost shap joblib matplotlib`  
- macOS XGBoost: `brew install libomp` then restart kernel


In [ ]:
%pip install -q pandas numpy scikit-learn xgboost shap joblib matplotlib


In [ ]:
from __future__ import annotations

import ctypes
import shutil
import subprocess
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)

BASE_DIR = Path("/Users/nikhilsingh.parihar/Documents/Megathon")
DATA_PATH = BASE_DIR / "green_customer_dataset/1_modelling_table/customer_model_table.csv"
OUT_DIR = BASE_DIR / "propensity_model_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCT = "ev_finance"  # ev_finance | green_home | solar | paperless
PRODUCT_MAP = {
    "ev_finance": {"label": "y_ev_finance_7d", "elig": "elig_ev_finance"},
    "green_home": {"label": "y_green_home_7d", "elig": "elig_green_home"},
    "solar": {"label": "y_solar_7d", "elig": "elig_solar"},
    "paperless": {"label": "y_paperless_7d", "elig": "elig_paperless"},
}
LABEL_COL = PRODUCT_MAP[PRODUCT]["label"]
ELIG_COL = PRODUCT_MAP[PRODUCT]["elig"]
ARTIFACT_PREFIX = OUT_DIR / PRODUCT


def _load_mac_libomp() -> bool:
    if sys.platform != "darwin":
        return False
    for lib_path in (
        Path("/opt/homebrew/opt/libomp/lib/libomp.dylib"),
        Path("/usr/local/opt/libomp/lib/libomp.dylib"),
    ):
        if lib_path.exists():
            ctypes.CDLL(str(lib_path))
            print("Loaded OpenMP:", lib_path)
            return True
    return False


if sys.platform == "darwin" and not _load_mac_libomp():
    brew = shutil.which("brew")
    if brew:
        print("Trying: brew install libomp ...")
        subprocess.run([brew, "install", "libomp"], check=False)
        _load_mac_libomp()

USE_XGB = True
try:
    import xgboost as xgb
    print("XGBoost", xgb.__version__)
except Exception as exc:
    USE_XGB = False
    from sklearn.ensemble import HistGradientBoostingClassifier
    print("XGBoost failed:", exc)
    print("Using HistGradientBoostingClassifier. For XGB: brew install libomp, restart kernel.")


In [ ]:
df = pd.read_csv(DATA_PATH)
print("rows", df.shape[0], "| label", LABEL_COL, "| rate", df[LABEL_COL].mean():.4f)


In [ ]:
label_cols = [c for c in df.columns if c.startswith("y_")]
elig_cols = [c for c in df.columns if c.startswith("elig_")]
drop_cols = {"customer_id", "split"} | set(label_cols) | set(elig_cols)

data = df[df[ELIG_COL] == 1].copy()
train_df = data[data["split"] == "train"].copy()
val_df = data[data["split"] == "val"].copy()
test_df = data[data["split"] == "test"].copy()
print("eligible train/val/test:", len(train_df), len(val_df), len(test_df))


In [ ]:
CAT_COLS = [
    "income_band", "region", "postcode_area",
    "digital_usage", "vehicle_type", "fuel_type",
]
NUM_COLS = [c for c in train_df.columns if c not in drop_cols and c not in CAT_COLS]


def build_feature_matrix(frame: pd.DataFrame, medians: pd.Series, columns: list[str]) -> pd.DataFrame:
    x = frame[NUM_COLS + CAT_COLS].copy()
    for col in NUM_COLS:
        x[col] = x[col].astype(float).fillna(medians[col])
    for col in CAT_COLS:
        x[col] = x[col].fillna("__MISSING__").astype(str)
    x = pd.get_dummies(x, columns=CAT_COLS, drop_first=False)
    return x.reindex(columns=columns, fill_value=0).astype(np.float64)


medians = train_df[NUM_COLS].median(numeric_only=True)
train_stub = train_df[NUM_COLS + CAT_COLS].copy()
for col in NUM_COLS:
    train_stub[col] = train_stub[col].astype(float).fillna(medians[col])
for col in CAT_COLS:
    train_stub[col] = train_stub[col].fillna("__MISSING__").astype(str)
FEATURE_COLUMNS = pd.get_dummies(train_stub, columns=CAT_COLS, drop_first=False).columns.tolist()

X_train = build_feature_matrix(train_df, medians, FEATURE_COLUMNS)
X_val = build_feature_matrix(val_df, medians, FEATURE_COLUMNS)
X_test = build_feature_matrix(test_df, medians, FEATURE_COLUMNS)

y_train = train_df[LABEL_COL].astype(int).to_numpy()
y_val = val_df[LABEL_COL].astype(int).to_numpy()
y_test = test_df[LABEL_COL].astype(int).to_numpy()
print("features:", X_train.shape[1])


In [ ]:
w = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

if USE_XGB:
    model = xgb.XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=w,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
else:
    model = HistGradientBoostingClassifier(
        max_iter=800,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
    )
    model.fit(X_train, y_train)

print("trained:", type(model).__name__)


In [ ]:
THRESHOLD = 0.5


def eval_split(name: str, y_true, y_prob) -> dict:
    pred = (y_prob >= THRESHOLD).astype(int)
    return {
        "split": name,
        "n": len(y_true),
        "positives": int(y_true.sum()),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]
p_test = model.predict_proba(X_test)[:, 1]

metrics_df = pd.DataFrame([
    eval_split("train", y_train, p_train),
    eval_split("val", y_val, p_val),
    eval_split("test", y_test, p_test),
])
display(metrics_df.round(4))


## Deciles (equal count on **validation**)

Sort validation customers by score (high → low), split into 10 equal-sized groups.  
Test customers are mapped with the **minimum validation score** in each decile.


In [ ]:
def equal_count_deciles(scores: np.ndarray) -> np.ndarray:
    """Decile 1 = top 10% by score (highest propensity)."""
    n = len(scores)
    order = np.argsort(-scores, kind="mergesort")
    out = np.empty(n, dtype=int)
    for d in range(10):
        lo = int(d * n / 10)
        hi = int((d + 1) * n / 10)
        out[order[lo:hi]] = d + 1
    return out


def decile_score_floors(reference_scores: np.ndarray) -> np.ndarray:
    ref_dec = equal_count_deciles(reference_scores)
    return np.array([reference_scores[ref_dec == d].min() for d in range(1, 11)])


def apply_score_floors(scores: np.ndarray, floors: np.ndarray) -> np.ndarray:
    out = np.full(len(scores), 10, dtype=int)
    for i, s in enumerate(scores):
        for d in range(1, 11):
            if s >= floors[d - 1]:
                out[i] = d
                break
    return out


def decile_table(y_true, scores, decile, split_name: str, base_rate: float) -> pd.DataFrame:
    rows = []
    for d in range(1, 11):
        m = decile == d
        rows.append({
            "split": split_name,
            "decile": d,
            "n": int(m.sum()),
            "conversion_rate": float(y_true[m].mean()) if m.any() else np.nan,
            "lift": (float(y_true[m].mean()) / base_rate) if m.any() and base_rate > 0 else np.nan,
            "score_min": float(scores[m].min()) if m.any() else np.nan,
            "score_max": float(scores[m].max()) if m.any() else np.nan,
        })
    return pd.DataFrame(rows)


val_base = y_val.mean()
val_decile = equal_count_deciles(p_val)
score_floors = decile_score_floors(p_val)
test_decile = apply_score_floors(p_test, score_floors)

val_deciles = decile_table(y_val, p_val, val_decile, "val", val_base)
test_deciles = decile_table(y_test, p_test, test_decile, "test", val_base)

print("Validation group sizes (should be ~equal):")
print(val_deciles[["decile", "n"]].to_string(index=False))
print("sum n =", val_deciles["n"].sum(), "| val rows =", len(y_val))

display(val_deciles)
display(test_deciles)

decile_confidence = val_deciles.set_index("decile")["conversion_rate"].to_dict()


In [ ]:
test_scored = test_df[["customer_id"]].copy()
test_scored["propensity_score"] = p_test
test_scored["decile"] = test_decile
test_scored["propensity_confidence"] = test_scored["decile"].map(decile_confidence)
test_scored["actual"] = y_test
test_scored.head(10)


In [ ]:
if USE_XGB:
    model_path = ARTIFACT_PREFIX.with_suffix(".json")
    model.save_model(model_path)
else:
    model_path = ARTIFACT_PREFIX.with_suffix(".joblib")
    joblib.dump(model, model_path)

preprocess_path = OUT_DIR / f"{PRODUCT}_preprocess.joblib"
floors_path = OUT_DIR / f"{PRODUCT}_decile_score_floors.npy"
confidence_path = OUT_DIR / f"{PRODUCT}_decile_confidence.json"

joblib.dump(
    {
        "feature_columns": FEATURE_COLUMNS,
        "num_cols": NUM_COLS,
        "cat_cols": CAT_COLS,
        "medians": medians,
        "label_col": LABEL_COL,
        "elig_col": ELIG_COL,
        "use_xgb": USE_XGB,
        "threshold": THRESHOLD,
    },
    preprocess_path,
)
np.save(floors_path, score_floors)
pd.Series(decile_confidence).to_json(confidence_path)
metrics_df.to_csv(OUT_DIR / f"{PRODUCT}_metrics.csv", index=False)
test_scored.to_csv(OUT_DIR / f"{PRODUCT}_test_scored.csv", index=False)

print("Saved model:", model_path)
print("Saved floors:", floors_path)


In [ ]:
SHAP_N = min(500, len(X_test))
idx = np.random.default_rng(42).choice(len(X_test), SHAP_N, replace=False)
X_shap = X_test.iloc[idx]

if USE_XGB:
    sv = shap.TreeExplainer(model).shap_values(X_shap)
    if isinstance(sv, list):
        sv = sv[1]
else:
    bg = shap.sample(X_train.to_numpy(), min(100, len(X_train)), random_state=42)
    sv = shap.Explainer(lambda x: model.predict_proba(x)[:, 1], bg)(X_shap.to_numpy()).values

imp = pd.DataFrame({"feature": X_shap.columns, "mean_abs_shap": np.abs(sv).mean(0)})
display(imp.sort_values("mean_abs_shap", ascending=False).head(20))

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_shap, show=False, max_display=20)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(y_test, p_test)
axes[0].plot(fpr, tpr, label=f"test AUC={roc_auc_score(y_test, p_test):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(title="ROC (test)", xlabel="FPR", ylabel="TPR")
axes[0].legend()
axes[0].grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(y_test, p_test)
axes[1].plot(rec, prec, label=f"test AP={average_precision_score(y_test, p_test):.3f}")
axes[1].axhline(y_test.mean(), color="k", ls=":", label=f"baseline {y_test.mean():.3f}")
axes[1].set(title="PR (test)", xlabel="Recall", ylabel="Precision")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
